# 06 Nested tuning, stacking, final model, interpretation, radar check and leave-one-city-out

Run the sections **in order and in one kernel**: sections 2 to 5 reuse the definitions of section 1. After a kernel
restart, rerun section 1 (it is resumable: finished outer folds are read from `outputs/tuning_8cities/parts/`).

| Section | What | Output folder (`outputs/`) |
|---|---|---|
| 1 | Nested spatial CV: HistGB (25 random-search candidates) and SVR (12), 3 inner folds, unweighted and inverse-slice weighted; mean ensemble and non-negative linear stacker fitted on the inner OOF predictions | `tuning_8cities/` |
| 2 | Final fit on all training rows, evaluation on the natural test set (block bootstrap CI), 187-band map variant | `final_model/` |
| 3 | Interpretation: permutation importance, SHAP, partial dependence | `interpretation/` |
| 4 | Radar check of the SVR | `radar_check_svr/` |
| 5 | Leave-one-city-out | `loco/` |

**Leakage rules:** scaling, imputation, feature selection and tuning happen inside the training folds; the natural test
set is touched only in section 2. Set `FAST = True` in section 1 to test the plumbing (then set it back to `False` before
section 2). Runtime on an Apple M-series laptop: tuning one HistGB outer fold took 21 to 86 min.

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

## 1. Nested CV, stacking (needs notebook 05 only for the optional paired comparison)

In [ ]:
# =====================================================================
# NESTED TUNING + STACKING CELL (CMP263): HistGB and SVR, each unweighted
# and with slice weights, tuned by nested spatial cross-validation, then
# combined. Resumable: every (model, weighting, outer fold) is saved as soon as it finishes.
#
# Design
#  * Outer loop = EXACTLY the spot-check folds (StratifiedGroupKFold, 5 folds,
#    seed 42, groups = city + 2,560 m block, strata = 5 height classes), so
#    every number here is paired with the spot-check on the same rows.
#  * Inner loop = 3 spatial folds built inside each outer training set only.
#    RandomizedSearchCV picks the hyper-parameters on the inner folds by
#    macro RMSE (the paper's selection metric); the outer fold only scores.
#  * The whole preprocessing (median imputer, scaler, Frantz step-1 RF
#    selection for SVR) sits INSIDE the pipeline, so it is refit on every
#    inner and outer training set: no validation row informs any step.
#  * Stacking: the meta-learner (non-negative linear regression with
#    intercept) is fit on INNER out-of-fold predictions of each outer
#    training set, then applied to the outer validation predictions.
#    The simple means need no fitting at all.
#  * Weighted variants use the spot-check scheme (inv_slice, cap 4, top
#    slice = 5 slices), computed on the outer training rows; inner folds get
#    the matching subset of those weights.  Scoring is always unweighted.
#  * Also writes folds.csv (sample_id -> outer fold) for the CNN.
#
# The natural test rows are read ONLY for their height histogram (the
# population frequencies of the weighted RMSE).  No test prediction here.
#
# =====================================================================
import os, json, time, warnings, inspect
import numpy as np, pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

# ------------------------- CONFIG ------------------------------------
DATA = config.OUT

TABLE = str(config.TABLE)
OUT = f"{DATA}/tuning_8cities"
SPOT_OOF = f"{DATA}/spot_check_8cities/oof_predictions_long.csv"      # for the paired comparison (optional)
MODELS = ["HistGB", "SVR"]
WEIGHTINGS = [False, True]       # False = unweighted, True = inv_slice weights (as in the spot-check weighted run)
N_ITER = {"HistGB": 25, "SVR": 12}   # random-search candidates per outer fold
INNER_FOLDS = 3
N_FOLDS, BLOCK_M, SEED = 5, 2560.0, 42   # must match the spot-check
W_MAX, TOP_EQUIV_SLICES = 4.0, 5
N_JOBS = -1                      # parallel candidates; models inside run single-threaded
FAST = False                     # True = tiny subsample, 3 candidates, few trees: tests the plumbing in minutes
# ---------------------------------------------------------------------

META = ["sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split"]
CLASS_EDGES = [2, 6, 15, 30]
CLASS_NAMES = ["<2 m", "2-6 m", "6-15 m", "15-30 m", ">30 m"]
FINE_EDGES = [2, 4, 6, 9, 12, 15, 20, 30, 50]
FINE_NAMES = ["<2", "2-4", "4-6", "6-9", "9-12", "12-15", "15-20", "20-30", "30-50", ">50"]
if FAST:
    OUT += "_FAST"
    N_ITER = {"HistGB": 3, "SVR": 3}


# ------------------- Frantz step 1 (same as the spot-check) ------------
class RFThreshold(BaseEstimator, TransformerMixin):
    """Frantz et al. (2021) supplement A2, step 1: RF on a random 10% subsample (500 trees, 1% of features
    per split); keep features with importance > 1 per mille (top `min_keep` if fewer survive)."""
    def __init__(self, frac=0.10, n_trees=500, mtry=0.01, thr=1e-3, min_keep=20, min_rows=500, seed=0, n_jobs=1):
        self.frac, self.n_trees, self.mtry, self.thr = frac, n_trees, mtry, thr
        self.min_keep, self.min_rows, self.seed, self.n_jobs = min_keep, min_rows, seed, n_jobs

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n = len(y)
        idx = rng.choice(n, size=min(n, max(int(self.frac * n), self.min_rows)), replace=False)
        rf = RandomForestRegressor(n_estimators=self.n_trees, max_features=self.mtry,
                                   n_jobs=self.n_jobs, random_state=self.seed).fit(X[idx], y[idx])
        imp = rf.feature_importances_
        keep = np.where(imp > self.thr)[0]
        if len(keep) < self.min_keep:
            keep = np.argsort(imp)[::-1][:self.min_keep]
        self.keep_ = np.sort(keep)
        return self

    def transform(self, X):
        return X[:, self.keep_]


# ------------------- pipelines and search spaces -----------------------
def make_pipeline(model):
    pre = [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
    if model == "HistGB":            # all 1,872 features, as in the spot-check
        est = HistGradientBoostingRegressor(early_stopping=False, random_state=SEED)
        return Pipeline(pre + [("model", est)])
    if model == "SVR":               # Frantz step 1, then RBF SVR on a standardised target
        sel = RFThreshold(n_trees=50 if FAST else 500, seed=SEED, n_jobs=1)
        est = TransformedTargetRegressor(regressor=SVR(kernel="rbf"), transformer=StandardScaler())
        return Pipeline(pre + [("sel", sel), ("model", est)])
    raise ValueError(model)


def search_space(model):
    if model == "HistGB":
        space = {"model__learning_rate": loguniform(0.02, 0.2),
                 "model__max_iter": randint(200, 801) if not FAST else randint(20, 41),
                 "model__max_leaf_nodes": [15, 31, 63, 127],
                 "model__min_samples_leaf": [10, 20, 50, 100],
                 "model__l2_regularization": loguniform(1e-3, 10)}
        if "max_features" in HistGradientBoostingRegressor().get_params():   # scikit-learn >= 1.4
            space["model__max_features"] = [0.3, 0.5, 1.0]
        return space
    if model == "SVR":                # gamma around 'scale' (about 1/195 on ~195 standardised features)
        return {"model__regressor__C": loguniform(1, 300),
                "model__regressor__gamma": loguniform(5e-4, 5e-2),
                "model__regressor__epsilon": uniform(0.02, 0.4)}       # in units of the target's SD
    raise ValueError(model)


# ------------------- weights and metrics -------------------------------
def slice_weights(y):
    """inv_slice weights, capped to [1, W_MAX], mean 1 (the spot-check scheme)."""
    y = np.asarray(y, float)
    s = np.where(y < 3, -1, np.minimum(np.floor(y).astype(int), 49))
    cnt = pd.Series(s).value_counts()
    ref = float(np.median([cnt.get(k, 0) for k in range(3, 30) if cnt.get(k, 0) > 0]))
    eff = cnt.astype(float)
    if 49 in eff.index:
        eff[49] = eff[49] / TOP_EQUIV_SLICES
    w = np.clip((ref / eff).reindex(s).to_numpy(), 1.0, W_MAX)
    w[s == -1] = 1.0
    return w / w.mean()


def macro_rmse(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    c = np.digitize(y, CLASS_EDGES)
    return float(np.mean([np.sqrt(np.mean((p[c == k] - y[c == k]) ** 2)) for k in np.unique(c)]))


MACRO_SCORER = make_scorer(lambda y, p: macro_rmse(y, np.clip(p, 0, None)), greater_is_better=False)


def population_fractions(te, cities):
    out = {}
    for c in cities:
        h = te.loc[te.city == c, "height_m"].to_numpy(float)
        if len(h):
            out[c] = np.bincount(np.digitize(h, FINE_EDGES), minlength=len(FINE_EDGES) + 1) / len(h)
    return out


def _wmse_city(y, p, f):
    fine = np.digitize(y, FINE_EDGES)
    mse, wt = [], []
    for b in range(len(FINE_EDGES) + 1):
        m = fine == b
        if m.any() and f[b] > 0:
            mse.append(np.mean((p[m] - y[m]) ** 2)); wt.append(f[b])
    return np.average(mse, weights=wt) if mse else np.nan


def metric_row(y, p, city, fr):
    e = p - y
    cls = np.digitize(y, CLASS_EDGES)
    row = dict(RMSE=float(np.sqrt(np.mean(e ** 2))), MAE=float(np.mean(np.abs(e))), RMSE_macro=macro_rmse(y, p))
    wm = [_wmse_city(y[city == c], p[city == c], fr[c]) for c in np.unique(city) if c in fr]
    row["RMSE_weighted"] = float(np.sqrt(np.nanmean(wm))) if wm else np.nan
    for k, nm in enumerate(CLASS_NAMES):
        m = cls == k
        row[f"RMSE {nm}"] = float(np.sqrt(np.mean(e[m] ** 2))) if m.any() else np.nan
    row["bias >=30 m"] = float(np.mean(e[y >= 30])) if (y >= 30).any() else np.nan
    return row


# ------------------- data and folds -------------------------------------
def load_table():
    df = config.read_table()
    feats = [c for c in df.columns if c not in META and c not in {"system:index", ".geo", "n_parts_50m", "slice_lo"}]
    df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
    tr = df[df.split == "train"].dropna(subset=["height_m"]).reset_index(drop=True)
    te = df[df.split == "test"].dropna(subset=["height_m"]).reset_index(drop=True)
    print(f"train rows {len(tr)} | test rows (histogram only) {len(te)} | {len(feats)} features")
    return tr, te, feats


def block_groups(d):
    gx, gy = np.floor(d.x_m / BLOCK_M).astype(int), np.floor(d.y_m / BLOCK_M).astype(int)
    return (d.city.astype(str) + "_" + gx.astype(str) + "_" + gy.astype(str)).to_numpy()


def outer_folds(y, groups):
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)   # = spot-check rep 0
    return list(cv.split(np.zeros(len(y)), np.digitize(y, CLASS_EDGES), groups))


def inner_folds(y, groups, k):
    cv = StratifiedGroupKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED + 100 + k)
    return list(cv.split(np.zeros(len(y)), np.digitize(y, CLASS_EDGES), groups))


# ------------------- one (model, weighting, outer fold) ----------------
def run_one(model, weighted, k, X, y, groups, itr, ite, sample_ids):
    tag = f"{model}{'_w' if weighted else ''}"
    path = f"{OUT}/parts/f{k}_{tag}"
    if os.path.exists(path + "_outer.csv"):
        return
    t0 = time.time()
    if FAST:
        itr = np.random.default_rng(SEED + k).choice(itr, size=min(len(itr), 900), replace=False)
    Xtr, ytr, gtr = X[itr], y[itr], groups[itr]
    inner = inner_folds(ytr, gtr, k)
    fit_kw = {"model__sample_weight": slice_weights(ytr)} if weighted else {}
    search = RandomizedSearchCV(make_pipeline(model), search_space(model), n_iter=N_ITER[model],
                                scoring=MACRO_SCORER, cv=inner, n_jobs=N_JOBS, random_state=SEED + k,
                                refit=True, error_score="raise")
    search.fit(Xtr, ytr, **fit_kw)
    best = search.best_estimator_
    pred_outer = np.clip(best.predict(X[ite]), 0, None)
    # inner out-of-fold predictions of the chosen configuration: training data for the stacker of this outer fold
    kw = "params" if "params" in inspect.signature(cross_val_predict).parameters else "fit_params"   # renamed in scikit-learn 1.4
    pred_inner = np.clip(cross_val_predict(clone(best), Xtr, ytr, cv=inner, n_jobs=N_JOBS, **{kw: fit_kw}), 0, None)
    nfeat = len(best.named_steps["sel"].keep_) if "sel" in best.named_steps else X.shape[1]
    pd.DataFrame({"sample_id": sample_ids[ite], "fold": k, "model": tag, "pred": pred_outer}).to_csv(path + "_outer.csv", index=False)
    pd.DataFrame({"sample_id": sample_ids[itr], "fold": k, "model": tag, "pred": pred_inner, "y": ytr}).to_csv(path + "_inner.csv", index=False)
    pd.DataFrame(search.cv_results_).drop(columns="params").to_csv(path + "_cv_results.csv", index=False)
    params = {kk: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v)
              for kk, v in search.best_params_.items()}
    json.dump(dict(model=tag, fold=k, best_params=params, inner_macro_rmse=-float(search.best_score_),
                   outer_macro_rmse=macro_rmse(y[ite], pred_outer), n_features=int(nfeat),
                   minutes=(time.time() - t0) / 60), open(path + "_best.json", "w"), indent=1)
    print(f"  fold {k} {tag:9s} inner macro {-search.best_score_:.2f} | outer macro {macro_rmse(y[ite], pred_outer):.2f} | "
          f"{nfeat} features | {(time.time()-t0)/60:.1f} min | {params}")


# ------------------- ensembles -----------------------------------------
def ensembles(outer, inner, y_of):
    """Simple means (no fitting) and a non-negative linear stacker fit per outer fold on INNER OOF predictions."""
    tags = sorted(outer.model.unique())
    O = outer.pivot_table(index=["sample_id", "fold"], columns="model", values="pred").reset_index()
    combos = {"mean HistGB+SVR": ["HistGB", "SVR"], "mean all four": ["HistGB", "HistGB_w", "SVR", "SVR_w"]}
    out = []
    for name, cols in combos.items():
        if all(c in tags for c in cols):
            out.append(pd.DataFrame({"sample_id": O.sample_id, "fold": O.fold, "model": name, "pred": O[cols].mean(axis=1)}))
    stack_w = []
    for name, cols in {"stack HistGB+SVR": ["HistGB", "SVR"], "stack all four": ["HistGB", "HistGB_w", "SVR", "SVR_w"]}.items():
        if not all(c in tags for c in cols):
            continue
        for k in sorted(O.fold.unique()):
            I = inner[inner.fold == k].pivot_table(index="sample_id", columns="model", values="pred")[cols].dropna()
            meta = LinearRegression(positive=True).fit(I.to_numpy(), y_of.reindex(I.index).to_numpy())
            Ok = O[O.fold == k]
            out.append(pd.DataFrame({"sample_id": Ok.sample_id, "fold": k, "model": name,
                                     "pred": np.clip(meta.predict(Ok[cols].to_numpy()), 0, None)}))
            stack_w.append(dict(stacker=name, fold=k, intercept=meta.intercept_, **dict(zip(cols, meta.coef_))))
    return (pd.concat(out, ignore_index=True) if out else pd.DataFrame()), pd.DataFrame(stack_w)


# ------------------- run ------------------------------------------------
def run_tuning():
    os.makedirs(f"{OUT}/parts", exist_ok=True)
    tr, te, feats = load_table()
    X = tr[feats].to_numpy(np.float32)
    y = tr.height_m.to_numpy(float)
    groups = block_groups(tr)
    sids = tr.sample_id.to_numpy()
    folds = outer_folds(y, groups)
    fold_of = np.empty(len(y), int)
    for k, (_, ite) in enumerate(folds):
        fold_of[ite] = k
    pd.DataFrame({"sample_id": sids, "city": tr.city, "fold": fold_of,
                  "block": groups}).to_csv(f"{OUT}/folds.csv", index=False)
    print(f"folds.csv written ({len(y)} rows; fold sizes {np.bincount(fold_of).tolist()})")
    for model in MODELS:
        for weighted in WEIGHTINGS:
            print(f"{model}{' weighted' if weighted else ''}: {N_ITER[model]} candidates x {INNER_FOLDS} inner folds per outer fold")
            for k, (itr, ite) in enumerate(folds):
                run_one(model, weighted, k, X, y, groups, itr, ite, sids)
    return summarise(tr, te)


def summarise(tr, te):
    fr = population_fractions(te, sorted(tr.city.unique()))
    lab = tr.set_index("sample_id")
    parts = os.listdir(f"{OUT}/parts")
    outer = pd.concat([pd.read_csv(f"{OUT}/parts/{f}") for f in parts if f.endswith("_outer.csv")], ignore_index=True)
    inner = pd.concat([pd.read_csv(f"{OUT}/parts/{f}") for f in parts if f.endswith("_inner.csv")], ignore_index=True)
    ens, stack_w = ensembles(outer, inner, lab.height_m)
    allp = pd.concat([outer, ens], ignore_index=True)
    allp["y"] = lab.height_m.reindex(allp.sample_id).to_numpy()
    allp["city"] = lab.city.reindex(allp.sample_id).to_numpy()
    allp.to_csv(f"{OUT}/oof_tuned_long.csv", index=False)
    pooled = pd.DataFrame([dict(model=m, **metric_row(g.y.to_numpy(), g.pred.to_numpy(), g.city.to_numpy(), fr))
                           for m, g in allp.groupby("model")]).set_index("model")
    perfold = pd.DataFrame([dict(model=m, fold=k, **metric_row(g.y.to_numpy(), g.pred.to_numpy(), g.city.to_numpy(), fr))
                            for (m, k), g in allp.groupby(["model", "fold"])])
    pooled["RMSE_macro_sd"] = perfold.groupby("model").RMSE_macro.std()
    pooled = pooled.sort_values("RMSE_macro")
    pooled.round(3).to_csv(f"{OUT}/summary_tuned.csv"); perfold.to_csv(f"{OUT}/fold_metrics_tuned.csv", index=False)
    best = [json.load(open(f"{OUT}/parts/{f}")) for f in parts if f.endswith("_best.json")]
    pd.DataFrame([{**{k: v for k, v in b.items() if k != "best_params"}, **b["best_params"]} for b in best]
                 ).sort_values(["model", "fold"]).to_csv(f"{OUT}/best_params.csv", index=False)
    if len(stack_w):
        stack_w.to_csv(f"{OUT}/stacker_weights.csv", index=False)
        print("\nStacker weights per outer fold (fit on inner OOF predictions):")
        print(stack_w.round(3).to_string(index=False))
    cols = ["RMSE_macro", "RMSE_macro_sd", "RMSE", "RMSE_weighted", "MAE"] + [f"RMSE {n}" for n in CLASS_NAMES] + ["bias >=30 m"]
    print("\nOuter-fold results, pooled over the 5 folds (m), sorted by macro RMSE:")
    print(pooled[cols].round(2).to_string())
    paired_vs_spot(tr)
    print(f"\nSaved in {OUT}: summary_tuned.csv, fold_metrics_tuned.csv, best_params.csv, oof_tuned_long.csv, "
          f"stacker_weights.csv, folds.csv")
    return dict(summary=pooled, folds=perfold, oof=allp, stack_weights=stack_w)


def paired_vs_spot(tr):
    """Per-fold macro RMSE of the tuned models minus the untuned spot-check models, same rows."""
    if not os.path.exists(SPOT_OOF) or FAST:
        print("\n(paired comparison with the spot-check skipped: no spot-check OOF file, or FAST mode)")
        return
    s = pd.read_csv(SPOT_OOF)
    s = s[(s.rep == 0) & s.model.isin(["HistGB", "SVR"])]
    s = s.merge(tr[["sample_id", "height_m"]], on="sample_id")
    tuned = pd.read_csv(f"{OUT}/oof_tuned_long.csv")
    for m in ["HistGB", "SVR"]:
        a, b = s[s.model == m], tuned[tuned.model == m]
        if a.empty or b.empty:
            continue
        same = all(set(a[a.fold == k].sample_id) == set(b[b.fold == k].sample_id) for k in range(N_FOLDS))
        if not same:
            print(f"\nWARNING: {m} spot-check and tuned folds hold different rows; paired comparison skipped")
            continue
        d = [macro_rmse(b[b.fold == k].y, b[b.fold == k].pred) - macro_rmse(a[a.fold == k].height_m, a[a.fold == k].pred)
             for k in range(N_FOLDS)]
        print(f"\n{m}: tuned minus spot-check macro RMSE per fold {np.round(d, 2).tolist()} "
              f"(mean {np.mean(d):+.2f} m; {sum(x < 0 for x in d)} of {N_FOLDS} folds better)")


results = run_tuning()

In [ ]:
# The folds written above must be the ones shipped in data/folds.csv (the CNN uses them too)
import pandas as pd
mine = pd.read_csv(f"{OUT}/folds.csv")
if config.FOLDS_CSV.exists():
    ship = pd.read_csv(config.FOLDS_CSV)
    same = mine[["sample_id", "fold"]].merge(ship[["sample_id", "fold"]], on="sample_id", suffixes=("", "_shipped"))
    print("rows", len(mine), "vs shipped", len(ship), "| same fold for", int((same.fold == same.fold_shipped).sum()), "of", len(same))
else:
    print("data/folds.csv not found; nothing to compare")

## 2. Final models and natural test evaluation

In [ ]:
# =====================================================================
# FINAL FIT + NATURAL TEST CELL (CMP263). Run AFTER the tuning cell in the same kernel: it reuses
# that cell's definitions (make_pipeline, search_space, MACRO_SCORER, load_table, block_groups,
# inner_folds, macro_rmse, CLASS_EDGES/NAMES, N_ITER, N_JOBS, SEED, DATA). After a kernel restart,
# just run the tuning cell again first: every fold is already saved, so it only re-summarises.
#
# What it does
#  1. Re-runs the same random search ONCE on all 16,954 training rows (3 spatial folds, macro RMSE)
#     to pick the final hyper-parameters of HistGB and SVR, refits both on all rows, saves them.
#     The final model is the plain mean of the two (the design chosen in cross-validation).
#  2. Scores it ONCE on the 8,800 natural test points: pooled, per city, per height class (with n),
#     on test points far from any training point (>= 100 m: no shared 50 m window; >= 330 m),
#     with bootstrap 95% intervals. Nothing is chosen on the test set.
#  3. Map variant: a HistGB refit on only the features SVR's Frantz step-1 selection kept, averaged
#     with SVR. A wall-to-wall map needs those ~200 bands per pixel instead of 1,872, so this is the
#     model the city map will use; its test cost is reported next to the full model. The band list
#     is written to selected_features.txt for the Earth Engine export.
#
# =====================================================================
import os, json, joblib, numpy as np, pandas as pd
from scipy.spatial import cKDTree
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
import matplotlib.pyplot as plt

assert "make_pipeline" in globals(), "run the tuning cell first in this kernel (its definitions are reused)"
assert not FAST, "set FAST = False in the tuning cell"

FINAL = f"{DATA}/final_model"
N_BOOT = 1000
FAR_M = [100, 330]
os.makedirs(FINAL, exist_ok=True)

tr, te, feats = load_table()
X, y = tr[feats].to_numpy(np.float32), tr.height_m.to_numpy(float)
groups = block_groups(tr)
Xte, yte, cte = te[feats].to_numpy(np.float32), te.height_m.to_numpy(float), te.city.to_numpy()
cv_all = inner_folds(y, groups, 99)            # 3 spatial folds over all training rows


# ---------------- 1. final hyper-parameters and fit -----------------------
def final_model(model):
    path = f"{FINAL}/{model}.joblib"
    if os.path.exists(path):
        print(f"{model}: loaded {path}")
        return joblib.load(path)
    search = RandomizedSearchCV(make_pipeline(model), search_space(model), n_iter=N_ITER[model], scoring=MACRO_SCORER,
                                cv=cv_all, n_jobs=N_JOBS, random_state=SEED, refit=True, error_score="raise")
    search.fit(X, y)
    est = search.best_estimator_
    params = {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v)
              for k, v in search.best_params_.items()}
    json.dump(dict(model=model, cv_macro_rmse=-float(search.best_score_), best_params=params),
              open(f"{FINAL}/{model}_params.json", "w"), indent=1)
    joblib.dump(est, path)
    print(f"{model}: CV macro {-search.best_score_:.2f} | {params}")
    return est

models = {m: final_model(m) for m in ["HistGB", "SVR"]}

# ---------------- map variant: HistGB on SVR's selected bands -------------
svr = models["SVR"]
kept = np.where(~np.isnan(svr.named_steps["imp"].statistics_))[0]       # columns the imputer kept
sel_idx = kept[svr.named_steps["sel"].keep_]
map_feats = [feats[i] for i in sel_idx]
open(f"{FINAL}/selected_features.txt", "w").write("\n".join(map_feats))
map_path = f"{FINAL}/HistGB_map.joblib"
if os.path.exists(map_path):
    hgb_map = joblib.load(map_path)
else:
    hp = {k.replace("model__", ""): v for k, v in models["HistGB"].named_steps["model"].get_params().items()}
    hgb_map = Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("model", HistGradientBoostingRegressor(**hp))]).fit(tr[map_feats].to_numpy(np.float32), y)
    joblib.dump(hgb_map, map_path)
print(f"map variant: {len(map_feats)} bands -> {FINAL}/selected_features.txt")

# ---------------- 2. test predictions (the only use of the test set) ----------------
P = {"HistGB": models["HistGB"].predict(Xte), "SVR": svr.predict(Xte)}
P["HistGB (map bands)"] = hgb_map.predict(te[map_feats].to_numpy(np.float32))
P = {k: np.clip(v, 0, None) for k, v in P.items()}
P["mean HistGB+SVR (final)"] = (P["HistGB"] + P["SVR"]) / 2
P["mean map variant"] = (P["HistGB (map bands)"] + P["SVR"]) / 2

# distance of each test point to the nearest TRAINING point of its city (same UTM zone per city)
trn = tr[["city", "x_m", "y_m"]]
d_eu, d_ch = np.full(len(te), np.inf), np.full(len(te), np.inf)
for c in np.unique(cte):
    pts = trn.loc[trn.city == c, ["x_m", "y_m"]].to_numpy()
    m = cte == c
    q = te.loc[m, ["x_m", "y_m"]].to_numpy()
    d_eu[m] = cKDTree(pts).query(q, k=1)[0]
    d_ch[m] = cKDTree(pts).query(q, k=1, p=np.inf)[0]
out = te[["sample_id", "city", "height_m", "x_m", "y_m"]].copy()
out["dist_to_train_m"], out["cheb_to_train_m"] = d_eu, d_ch
for k, v in P.items():
    out[k] = v
out.to_csv(f"{FINAL}/test_predictions.csv", index=False)


def scores(yv, pv):
    e = pv - yv
    cls = np.digitize(yv, CLASS_EDGES)
    r = dict(n=len(yv), RMSE=np.sqrt(np.mean(e ** 2)), MAE=np.mean(np.abs(e)), bias=np.mean(e),
             R2=1 - np.sum(e ** 2) / np.sum((yv - yv.mean()) ** 2), RMSE_macro=macro_rmse(yv, pv))
    for k, nm in enumerate(CLASS_NAMES):
        mk = cls == k
        r[f"n {nm}"] = int(mk.sum())
        r[f"RMSE {nm}"] = np.sqrt(np.mean(e[mk] ** 2)) if mk.any() else np.nan
    return r


rows = []
for name, p in P.items():
    rows.append(dict(model=name, subset="all", **scores(yte, p)))
    for c in np.unique(cte):
        m = cte == c
        rows.append(dict(model=name, subset=c, **scores(yte[m], p[m])))
    for f in FAR_M:
        m = d_eu >= f
        rows.append(dict(model=name, subset=f">= {f} m from training", **scores(yte[m], p[m])))
res = pd.DataFrame(rows)
res.round(3).to_csv(f"{FINAL}/test_metrics.csv", index=False)

# bootstrap 95% intervals (rows resampled within each city)
rng = np.random.default_rng(SEED)
idx_city = [np.where(cte == c)[0] for c in np.unique(cte)]
boot = {k: [] for k in ["mean HistGB+SVR (final)", "HistGB", "SVR", "mean map variant"]}
for _ in range(N_BOOT):
    ii = np.concatenate([rng.choice(ix, len(ix)) for ix in idx_city])
    for k in boot:
        e = P[k][ii] - yte[ii]
        tall = yte[ii] >= 30
        boot[k].append((np.sqrt(np.mean(e ** 2)), macro_rmse(yte[ii], P[k][ii]),
                        np.sqrt(np.mean(e[tall] ** 2)) if tall.any() else np.nan))
ci = pd.DataFrame([dict(model=k, **{f"{s} [2.5%, 97.5%]": f"[{np.nanpercentile(np.array(v)[:, j], 2.5):.2f}, "
                                     f"{np.nanpercentile(np.array(v)[:, j], 97.5):.2f}]"
                                     for j, s in enumerate(["RMSE", "RMSE_macro", "RMSE >30 m"])})
                   for k, v in boot.items()])
ci.to_csv(f"{FINAL}/test_bootstrap_ci.csv", index=False)

# ---------------- print ------------------------------------------------------
cols = ["n", "RMSE", "MAE", "bias", "R2", "RMSE_macro"] + [f"RMSE {n}" for n in CLASS_NAMES]
print("\nNATURAL TEST SET, all cities (m):")
print(res[res.subset == "all"].set_index("model")[cols].round(2).to_string())
print("\nclass sizes in the test set:", {n: int(res.loc[(res.subset == "all"), f"n {n}"].iloc[0]) for n in CLASS_NAMES},
      "\n(the >30 m class is small: read its RMSE with the bootstrap interval, not alone)")
print("\nFinal model per city (m):")
print(res[(res.model == "mean HistGB+SVR (final)") & ~res.subset.isin(["all"])].set_index("subset")[cols].round(2).to_string())
print("\nBootstrap 95% intervals:")
print(ci.to_string(index=False))
print(f"\nTest points >= 100 m / >= 330 m from any training point: {(d_eu >= 100).mean():.0%} / {(d_eu >= 330).mean():.0%}")

# ---------------- figure: prediction vs reference on the test set ----------------
p = P["mean HistGB+SVR (final)"]
fig, ax = plt.subplots(figsize=(5.2, 5))
hb = ax.hexbin(yte, p, gridsize=45, bins="log", cmap="Greys", mincnt=1, extent=(0, 80, 0, 80))
ax.plot([0, 80], [0, 80], color="#2a6fdb", lw=1.2, label="1:1")
ax.set_xlim(0, 80); ax.set_ylim(0, 80); ax.set_aspect("equal")
ax.set_xlabel("reference height (m)"); ax.set_ylabel("predicted height (m)")
r_all = res[(res.model == "mean HistGB+SVR (final)") & (res.subset == "all")].iloc[0]
ax.set_title(f"Natural test set, 8 cities (n = {int(r_all.n):,})\nRMSE {r_all.RMSE:.2f} m, MAE {r_all.MAE:.2f} m, "
             f"bias {r_all.bias:+.2f} m", fontsize=10)
ax.legend(loc="upper left", frameon=False)
fig.colorbar(hb, ax=ax, shrink=0.8, label="test points (log)")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FINAL}/test_scatter.png", dpi=200)
plt.show()
print(f"\nSaved in {FINAL}: HistGB.joblib, SVR.joblib, HistGB_map.joblib, *_params.json, selected_features.txt, "
      f"test_predictions.csv, test_metrics.csv, test_bootstrap_ci.csv, test_scatter.png")

## 3. Interpretation

In [ ]:
# =====================================================================
# INTERPRETATION CELL (CMP263): what drives the final model (mean of HistGB and SVR).
# Loads the saved final models; nothing is retrained.
#
#  1. Grouped permutation importance of the FINAL MEAN MODEL. The 1,872 features are near-duplicates
#     of each other (TCW_p25 and TCW_p25_OPN, ...), so single-feature permutation would understate
#     them: each GROUP is permuted jointly (the same row shuffle for all its columns). Three
#     groupings: base layer (VV, VH, the ten bands, the six indices), statistic (mean, p10, ...),
#     morphological operator (none, ERO, DIL, ...). Score = increase of macro RMSE and of RMSE,
#     mean and SD over N_REPEATS shuffles.
#       - on the natural TEST set (held out; dominated by low buildings), and
#       - on a height-stratified subsample of the TRAINING rows (all height classes, incl. tall;
#         the model was fit on these rows, so importances there can be inflated: read as
#         "what the model relies on", not as generalisation).
#  2. SHAP (TreeExplainer) for HistGB on a stratified training subsample: mean |SHAP| per feature
#     and per group, for all rows and for rows of 30 m and more (what drives tall predictions).
#     Optional: skipped if the shap package is missing (pip install shap).
#  3. Partial dependence (+ ICE lines) of the mean model, and of HistGB and SVR separately, for the
#     top features (by SHAP, or by HistGB permutation if shap is missing).
#  4. Comparison with Frantz et al. (2021) supplement Table A2 (their 50 selected features).
#
# =====================================================================
import os, numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
from joblib import Parallel, delayed

DATA = config.OUT
TABLE = str(config.TABLE)
FINAL = f"{DATA}/final_model"
OUT = f"{DATA}/interpretation"
N_REPEATS = 5
N_TRAIN_SUB = 6000            # stratified training subsample for permutation importance
N_SHAP = 2000
N_PDP, N_ICE, N_GRID, TOP_PDP = 800, 60, 20, 6
SEED, N_JOBS = 42, -1
CLASS_EDGES = [2, 6, 15, 30]
os.makedirs(OUT, exist_ok=True)

if "RFThreshold" not in globals():        # needed to load SVR.joblib; same code as the tuning cell
    from sklearn.base import BaseEstimator, TransformerMixin
    from sklearn.ensemble import RandomForestRegressor

    class RFThreshold(BaseEstimator, TransformerMixin):
        def __init__(self, frac=0.10, n_trees=500, mtry=0.01, thr=1e-3, min_keep=20, min_rows=500, seed=0, n_jobs=1):
            self.frac, self.n_trees, self.mtry, self.thr = frac, n_trees, mtry, thr
            self.min_keep, self.min_rows, self.seed, self.n_jobs = min_keep, min_rows, seed, n_jobs

        def fit(self, X, y):
            rng = np.random.default_rng(self.seed)
            idx = rng.choice(len(y), size=min(len(y), max(int(self.frac * len(y)), self.min_rows)), replace=False)
            rf = RandomForestRegressor(n_estimators=self.n_trees, max_features=self.mtry, n_jobs=self.n_jobs,
                                       random_state=self.seed).fit(X[idx], y[idx])
            keep = np.where(rf.feature_importances_ > self.thr)[0]
            if len(keep) < self.min_keep:
                keep = np.argsort(rf.feature_importances_)[::-1][:self.min_keep]
            self.keep_ = np.sort(keep)
            return self

        def transform(self, X):
            return X[:, self.keep_]


def macro_rmse(y, p):
    c = np.digitize(y, CLASS_EDGES)
    return float(np.mean([np.sqrt(np.mean((p[c == k] - y[c == k]) ** 2)) for k in np.unique(c)]))


# ---------------- data and models ----------------
META_COLS = {"sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split",
             "system:index", ".geo", "n_parts_50m", "slice_lo"}
df = config.read_table()
feats = [c for c in df.columns if c not in META_COLS]
df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
tr = df[df.split == "train"].dropna(subset=["height_m"]).reset_index(drop=True)
te = df[df.split == "test"].dropna(subset=["height_m"]).reset_index(drop=True)
hgb, svr = joblib.load(f"{FINAL}/HistGB.joblib"), joblib.load(f"{FINAL}/SVR.joblib")
kept = np.where(~np.isnan(svr.named_steps["imp"].statistics_))[0]
svr_cols = set(kept[svr.named_steps["sel"].keep_].tolist())          # columns SVR actually uses
print(f"{len(feats)} features; SVR uses {len(svr_cols)}; HistGB uses all")


def strat_sub(d, n, seed):
    cls = np.digitize(d.height_m, CLASS_EDGES)
    per = int(np.ceil(n / len(np.unique(cls))))
    rng = np.random.default_rng(seed)
    idx = np.concatenate([rng.choice(np.where(cls == k)[0], min(per, (cls == k).sum()), replace=False)
                          for k in np.unique(cls)])
    return d.iloc[np.sort(idx)].reset_index(drop=True)


sets = {"test (natural)": te, "train (stratified subsample)": strat_sub(tr, N_TRAIN_SUB, SEED)}

# ---------------- groups ----------------
def parts(f):
    p = f.split("_")
    return p[0], p[1], (p[2] if len(p) > 2 else "none")


GROUPINGS = {"base layer": 0, "statistic": 1, "operator": 2}
groups = {}
for gname, pos in GROUPINGS.items():
    g = {}
    for j, f in enumerate(feats):
        g.setdefault(parts(f)[pos], []).append(j)
    groups[gname] = g


def mean_pred(X, svr_pred=None):
    ph = np.clip(hgb.predict(X), 0, None)
    ps = np.clip(svr.predict(X), 0, None) if svr_pred is None else svr_pred
    return (ph + ps) / 2, ph, ps


def one_perm(X, y, cols, base_svr, seed):
    rng = np.random.default_rng(seed)
    Xp = X.copy()
    Xp[:, cols] = X[rng.permutation(len(X))][:, cols]
    touches = any(c in svr_cols for c in cols)
    p = mean_pred(Xp, None if touches else base_svr)[0]
    return macro_rmse(y, p), float(np.sqrt(np.mean((p - y) ** 2)))


rows = []
for sname, d in sets.items():
    X, y = d[feats].to_numpy(np.float32), d.height_m.to_numpy(float)
    p0, _, s0 = mean_pred(X)
    m0, r0 = macro_rmse(y, p0), float(np.sqrt(np.mean((p0 - y) ** 2)))
    print(f"\n{sname}: n = {len(y)}, baseline macro RMSE {m0:.2f} m, RMSE {r0:.2f} m; permuting ...")
    jobs = [(gname, key, cols, k) for gname, g in groups.items() for key, cols in g.items() for k in range(N_REPEATS)]
    res = Parallel(n_jobs=N_JOBS)(delayed(one_perm)(X, y, cols, s0, SEED + 1000 * k + i)
                                  for i, (gname, key, cols, k) in enumerate(jobs))
    tmp = pd.DataFrame([dict(set=sname, grouping=g, group=key, n_features=len(c), d_macro=m - m0, d_rmse=r - r0)
                        for (g, key, c, k), (m, r) in zip(jobs, res)])
    rows.append(tmp.groupby(["set", "grouping", "group", "n_features"]).agg(
        d_macro=("d_macro", "mean"), d_macro_sd=("d_macro", "std"),
        d_rmse=("d_rmse", "mean"), d_rmse_sd=("d_rmse", "std")).reset_index())
perm = pd.concat(rows).sort_values(["set", "grouping", "d_macro"], ascending=[True, True, False])
perm.round(4).to_csv(f"{OUT}/permutation_importance_groups.csv", index=False)
for sname in sets:
    print(f"\nGrouped permutation importance, {sname}: increase of macro RMSE / RMSE (m), mean of {N_REPEATS}")
    for gname in GROUPINGS:
        t = perm[(perm.set == sname) & (perm.grouping == gname)].head(8)
        print(f"  by {gname}: " + "; ".join(f"{r.group} {r.d_macro:+.2f}/{r.d_rmse:+.2f}" for r in t.itertuples()))

fig, axs = plt.subplots(len(sets), 3, figsize=(13, 3.2 * len(sets) + 1.5), squeeze=False)
for i, sname in enumerate(sets):
    for j, gname in enumerate(GROUPINGS):
        t = perm[(perm.set == sname) & (perm.grouping == gname)].sort_values("d_macro").tail(12)
        ax = axs[i, j]
        ax.barh(t.group, t.d_macro, xerr=t.d_macro_sd, color="#8c8c8c", ecolor="#444", height=0.6)
        ax.axvline(0, color="#444", lw=0.8)
        ax.set_title(f"{sname}: by {gname}", fontsize=9)
        ax.set_xlabel("increase of macro RMSE when permuted (m)", fontsize=8)
        ax.tick_params(labelsize=8)
        for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/permutation_importance_groups.png", dpi=200); plt.show()

# ---------------- SHAP for HistGB ----------------
top_feats = None
try:
    import shap
    sub = strat_sub(tr, N_SHAP, SEED + 1)
    Xs = hgb[:-1].transform(sub[feats].to_numpy(np.float32))       # imputer + scaler of the pipeline
    sv = shap.TreeExplainer(hgb.named_steps["model"]).shap_values(Xs)
    tall = sub.height_m.to_numpy() >= 30
    sh = pd.DataFrame({"feature": feats, "mean_abs_shap_all": np.abs(sv).mean(0),
                       "mean_abs_shap_tall": np.abs(sv[tall]).mean(0) if tall.any() else np.nan,
                       "used_by_svr": [j in svr_cols for j in range(len(feats))]})
    for gname, pos in GROUPINGS.items():
        sh[gname] = [parts(f)[pos] for f in feats]
    sh = sh.sort_values("mean_abs_shap_all", ascending=False)
    sh.to_csv(f"{OUT}/shap_histgb.csv", index=False)
    print("\nHistGB, top 15 features by mean |SHAP| (all rows | rows >= 30 m), m:")
    print(sh.head(15)[["feature", "mean_abs_shap_all", "mean_abs_shap_tall", "used_by_svr"]].round(3).to_string(index=False))
    for gname in GROUPINGS:
        g = sh.groupby(gname)[["mean_abs_shap_all", "mean_abs_shap_tall"]].sum()
        g = (g / g.sum()).sort_values("mean_abs_shap_all", ascending=False).head(8)
        print(f"  share of total |SHAP| by {gname} (all | tall): " +
              "; ".join(f"{k} {a:.0%}|{b:.0%}" for k, (a, b) in g.iterrows()))
    plt.figure()
    shap.summary_plot(sv, Xs, feature_names=feats, max_display=15, show=False)
    plt.title("HistGB: SHAP values (standardised feature values)", fontsize=10)
    plt.tight_layout(); plt.savefig(f"{OUT}/shap_histgb_beeswarm.png", dpi=200); plt.show()
    top_feats = sh[sh.used_by_svr].feature.head(TOP_PDP).tolist()      # top features that both models use
except ImportError:
    print("\nshap not installed: SHAP skipped (pip install shap). PDP features ranked by HistGB permutation instead.")

if top_feats is None:                     # fallback ranking: single-feature permutation of HistGB on the test set
    X, y = te[feats].to_numpy(np.float32), te.height_m.to_numpy(float)
    base = macro_rmse(y, np.clip(hgb.predict(X), 0, None))
    rng = np.random.default_rng(SEED)
    sc = {}
    for j in sorted(svr_cols):
        Xp = X.copy(); Xp[:, j] = X[rng.permutation(len(X)), j]
        sc[feats[j]] = macro_rmse(y, np.clip(hgb.predict(Xp), 0, None)) - base
    top_feats = sorted(sc, key=sc.get, reverse=True)[:TOP_PDP]

# ---------------- partial dependence (+ ICE) ----------------
sub = strat_sub(tr, N_PDP, SEED + 2)
Xb = sub[feats].to_numpy(np.float32)
rng = np.random.default_rng(SEED)
ice_rows = rng.choice(len(Xb), min(N_ICE, len(Xb)), replace=False)
nc = 3
fig, axs = plt.subplots(int(np.ceil(len(top_feats) / nc)), nc, figsize=(12, 3.4 * np.ceil(len(top_feats) / nc)), squeeze=False)
pd_rows = []
for k, f in enumerate(top_feats):
    j = feats.index(f)
    grid = np.unique(np.nanpercentile(tr[f], np.linspace(2, 98, N_GRID)))
    mm, hh, ss, ice = [], [], [], []
    for v in grid:
        Xv = Xb.copy(); Xv[:, j] = v
        p, ph, ps = mean_pred(Xv)
        mm.append(p.mean()); hh.append(ph.mean()); ss.append(ps.mean()); ice.append(p[ice_rows])
        pd_rows.append(dict(feature=f, value=v, mean_model=p.mean(), HistGB=ph.mean(), SVR=ps.mean()))
    ax = axs.flat[k]
    ax.plot(grid, np.array(ice), color="#bbbbbb", lw=0.5, alpha=0.6)
    ax.plot(grid, mm, color="#2a6fdb", lw=2.2, label="mean model")
    ax.plot(grid, hh, color="#444", lw=1.2, ls="--", label="HistGB")
    ax.plot(grid, ss, color="#444", lw=1.2, ls=":", label="SVR")
    ax.set_title(f, fontsize=9); ax.set_xlabel("feature value", fontsize=8); ax.set_ylabel("predicted height (m)", fontsize=8)
    ax.tick_params(labelsize=8)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    if k == 0: ax.legend(fontsize=7, frameon=False)
for ax in list(axs.flat)[len(top_feats):]: ax.axis("off")
fig.suptitle("Partial dependence (lines) and ICE (grey, 60 rows) on a height-stratified training subsample", fontsize=10)
fig.tight_layout(); fig.savefig(f"{OUT}/partial_dependence.png", dpi=200); plt.show()
pd.DataFrame(pd_rows).to_csv(f"{OUT}/partial_dependence.csv", index=False)

# ---------------- comparison with Frantz et al. (2021), supplement Table A2 ----------------
# Their 50 selected features as printed in Table A2 (Blue=B2, Green=B3, RED=B4, RE1-3=B5-B7, NIR=B8, NIR2=B8A,
# TC Green=TCG, TC Bright=TCB; the table lists some features more than once, so there are 45 distinct ones).
FRANTZ_A2 = ['VH_mean_OPN', 'B4_p90_ERO', 'B8A_p100_BHT', 'TCG_p50_CLS', 'VV_p10_OPN', 'B7_p75_ERO', 'B8A_p90_ERO',
             'VV_mean_CLS', 'B2_p0_GRD', 'B3_p0_CLS', 'TCG_mean_OPN', 'TCG_p0_CLS', 'B6_p75_ERO', 'B6_p90_OPN',
             'B2_p25_CLS', 'TCB_p25_ERO', 'B8_mean_OPN', 'B7_mean_ERO', 'B3_mean_ERO', 'B6_p10_OPN', 'B8_p10_OPN',
             'VV_p75_CLS', 'VV_p75_CLS', 'mNDWI_p25_GRD', 'VV_kurtosis_OPN', 'VH_mean_CLS', 'VH_p25_DIL', 'B2_p0_DIL',
             'B8_p90_OPN', 'B2_p25_DIL', 'TCG_p25_ERO', 'B8A_p75_ERO', 'B6_p10_ERO', 'VH_mean_DIL', 'VV_kurtosis_OPN',
             'VH_p10_CLS', 'VV_p0_DIL', 'B8_p0_ERO', 'B8A_p90_GRD', 'B6_range_OPN', 'VH_IQR_DIL', 'VV_kurtosis_OPN',
             'mNDWI_p25_GRD', 'B8_p10_ERO', 'B3_p0_CLS', 'B3_mean_OPN', 'VH_p25_CLS', 'B6_p100_ERO', 'B4_p75_CLS',
             'B3_p0_GRD']
DOMAIN = {"VV": "radar", "VH": "radar", "B2": "visible", "B3": "visible", "B4": "visible", "B5": "red edge",
          "B6": "red edge", "B7": "red edge", "B8": "NIR", "B8A": "NIR", "B11": "SWIR", "B12": "SWIR",
          "NDVI": "index (no SWIR)", "TCG": "Tasseled Cap", "TCB": "Tasseled Cap", "TCW": "Tasseled Cap",
          "NDBI": "index (SWIR)", "mNDWI": "index (SWIR)"}
ours = [feats[j] for j in sorted(svr_cols)]
fr = sorted(set(FRANTZ_A2))
comp = pd.DataFrame({
    "ours: SVR selection (%)": pd.Series([DOMAIN[parts(f)[0]] for f in ours]).value_counts(normalize=True) * 100,
    "Frantz Table A2 (%)": pd.Series([DOMAIN[parts(f)[0]] for f in fr]).value_counts(normalize=True) * 100}).fillna(0)
comp_op = pd.DataFrame({
    "ours: SVR selection (%)": pd.Series([parts(f)[2] for f in ours]).value_counts(normalize=True) * 100,
    "Frantz Table A2 (%)": pd.Series([parts(f)[2] for f in fr]).value_counts(normalize=True) * 100}).fillna(0)
print(f"\nFeature domains: our {len(ours)} SVR features vs Frantz's {len(fr)} distinct Table A2 features")
print(comp.round(0).sort_values("Frantz Table A2 (%)", ascending=False).to_string())
print(comp_op.round(0).to_string())
print("features in both lists:", sorted(set(ours) & set(fr)) or "none")
comp.round(1).to_csv(f"{OUT}/domains_vs_frantz.csv"); comp_op.round(1).to_csv(f"{OUT}/operators_vs_frantz.csv")
print(f"\nSaved in {OUT}: permutation_importance_groups.csv/.png, shap_histgb.csv, shap_histgb_beeswarm.png, "
      f"partial_dependence.csv/.png, domains_vs_frantz.csv, operators_vs_frantz.csv")

## 4. Radar check (SVR)

In [ ]:
# =====================================================================
# SENSOR COMPARISON WITH SVR (CMP263): the Frantz et al. (2021) Fig. 6 comparison with the learner
# they used. Self-contained notebook cell; resumable (each feature set and fold is saved).
#
# Frantz et al. trained one SVR per feature set (radar, optical, both), each tuned by grid search.
# Here, for each feature set, the SVR pipeline of our tuning cell (median imputer, scaler, Frantz
# step-1 RF selection, RBF SVR on a standardised target) is tuned by the SAME nested random search
# (12 candidates, 3 inner spatial folds, macro RMSE) inside each of the 5 outer folds (folds.csv).
# Tuning each set separately matters: the settings of the combined model (gamma in particular)
# are not right for 208 radar features.
# Then, within each city: radar-only and optical-only SVR trained and validated inside the city on
# the same folds, with the median of that feature set's tuned settings (no inner search per city,
# to keep the run short; stated as a simplification).
#
# Differences from Frantz to state: they used the full feature space for Fig. 6 (no selection), one
# city (Berlin) and a random 90/10 split; we use our step-1 selection, eight cities and spatial folds.
#
# =====================================================================
import os, json, time, numpy as np, pandas as pd
from scipy.stats import loguniform, uniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

DATA = config.OUT
TABLE = str(config.TABLE)
FOLDS = f"{DATA}/tuning_8cities/folds.csv"
OUT = f"{DATA}/radar_check_svr"
N_ITER, INNER_FOLDS, SEED, N_JOBS = 12, 3, 42, -1
CLASS_EDGES = [2, 6, 15, 30]
CLASS_NAMES = ["<2 m", "2-6 m", "6-15 m", "15-30 m", ">30 m"]
os.makedirs(f"{OUT}/parts", exist_ok=True)


class RFThreshold(BaseEstimator, TransformerMixin):
    """Frantz et al. (2021) supplement A2, step 1 (same code as the tuning cell)."""
    def __init__(self, frac=0.10, n_trees=500, mtry=0.01, thr=1e-3, min_keep=20, min_rows=500, seed=0, n_jobs=1):
        self.frac, self.n_trees, self.mtry, self.thr = frac, n_trees, mtry, thr
        self.min_keep, self.min_rows, self.seed, self.n_jobs = min_keep, min_rows, seed, n_jobs

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        idx = rng.choice(len(y), size=min(len(y), max(int(self.frac * len(y)), self.min_rows)), replace=False)
        rf = RandomForestRegressor(n_estimators=self.n_trees, max_features=self.mtry, n_jobs=self.n_jobs,
                                   random_state=self.seed).fit(X[idx], y[idx])
        keep = np.where(rf.feature_importances_ > self.thr)[0]
        if len(keep) < self.min_keep:
            keep = np.argsort(rf.feature_importances_)[::-1][:self.min_keep]
        self.keep_ = np.sort(keep)
        return self

    def transform(self, X):
        return X[:, self.keep_]


def macro_rmse(y, p):
    c = np.digitize(y, CLASS_EDGES)
    return float(np.mean([np.sqrt(np.mean((p[c == k] - y[c == k]) ** 2)) for k in np.unique(c)]))


def scores(y, p):
    e = p - y
    c = np.digitize(y, CLASS_EDGES)
    r = dict(n=len(y), RMSE_macro=macro_rmse(y, p), RMSE=float(np.sqrt(np.mean(e ** 2))), bias=float(e.mean()))
    for k, nm in enumerate(CLASS_NAMES):
        r[f"RMSE {nm}"] = float(np.sqrt(np.mean(e[c == k] ** 2))) if (c == k).any() else np.nan
    return r


MACRO = make_scorer(lambda y, p: macro_rmse(y, np.clip(p, 0, None)), greater_is_better=False)
SPACE = {"model__regressor__C": loguniform(1, 300), "model__regressor__gamma": loguniform(5e-4, 5e-2),
         "model__regressor__epsilon": uniform(0.02, 0.4)}


def svr_pipe(**p):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                     ("sel", RFThreshold(n_trees=500, seed=SEED, n_jobs=1)),
                     ("model", TransformedTargetRegressor(regressor=SVR(kernel="rbf", **p), transformer=StandardScaler()))])


META_COLS = {"sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split",
             "system:index", ".geo", "n_parts_50m", "slice_lo"}
df = config.read_table()
feats = [c for c in df.columns if c not in META_COLS]
df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
tr = df[df.split == "train"].dropna(subset=["height_m"]).reset_index(drop=True)
del df
tr = tr.merge(pd.read_csv(FOLDS)[["sample_id", "fold"]], on="sample_id", how="left")
assert tr.fold.notna().all(), "some training rows have no fold in folds.csv"
y, fold, city = tr.height_m.to_numpy(float), tr.fold.to_numpy(int), tr.city.to_numpy()
gx, gy = np.floor(tr.x_m / 2560.0).astype(int), np.floor(tr.y_m / 2560.0).astype(int)
groups = (tr.city.astype(str) + "_" + gx.astype(str) + "_" + gy.astype(str)).to_numpy()
radar = [f for f in feats if f.split("_")[0] in ("VV", "VH")]
SETS = {"radar only": radar, "optical only": [f for f in feats if f not in radar], "all": feats}

# ---------------- 1. pooled, nested-tuned SVR per feature set ----------------
best = {}
for name, cols in SETS.items():
    X = tr[cols].to_numpy(np.float32)
    for k in np.unique(fold):
        part = f"{OUT}/parts/{name.replace(' ', '_')}_f{k}.csv"
        if os.path.exists(part):
            continue
        t0 = time.time()
        itr, ite = np.where(fold != k)[0], np.where(fold == k)[0]
        inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=SEED + 100 + k)
                     .split(np.zeros(len(itr)), np.digitize(y[itr], CLASS_EDGES), groups[itr]))
        s = RandomizedSearchCV(svr_pipe(), SPACE, n_iter=N_ITER, scoring=MACRO, cv=inner, n_jobs=N_JOBS,
                               random_state=SEED + k, refit=True).fit(X[itr], y[itr])
        p = np.clip(s.best_estimator_.predict(X[ite]), 0, None)
        prm = {kk.split("__")[-1]: float(v) for kk, v in s.best_params_.items()}
        pd.DataFrame({"sample_id": tr.sample_id.iloc[ite], "fold": k, "pred": p,
                      "n_features": len(s.best_estimator_.named_steps["sel"].keep_), **prm}).to_csv(part, index=False)
        print(f"  {name:13s} fold {k}: inner macro {-s.best_score_:.2f} | outer macro {macro_rmse(y[ite], p):.2f} | "
              f"{len(s.best_estimator_.named_steps['sel'].keep_)} features | {prm} | {(time.time() - t0) / 60:.1f} min")

oof, rows = {}, []
for name in SETS:
    d = pd.concat([pd.read_csv(f"{OUT}/parts/{name.replace(' ', '_')}_f{k}.csv") for k in np.unique(fold)])
    d = d.set_index("sample_id").reindex(tr.sample_id)
    oof[name] = d.pred.to_numpy()
    best[name] = {p: float(d[p].median()) for p in ["C", "gamma", "epsilon"]}
    rows.append(dict(features=name, n_features_median=int(d.n_features.median()), **scores(y, oof[name]),
                     **{f"median {p}": v for p, v in best[name].items()}))
pooled = pd.DataFrame(rows).set_index("features")
pooled.round(4).to_csv(f"{OUT}/pooled_cv_svr.csv")
print("\n1. Nested-tuned SVR, pooled 5-fold spatial CV (m). Frantz Berlin SVR RMSE: radar 7.47, optical 7.03, both 6.58")
print(pooled.round(3).to_string())
r, o, a = pooled.loc["radar only", "RMSE"], pooled.loc["optical only", "RMSE"], pooled.loc["all", "RMSE"]
print(f"radar/optical RMSE ratio {r / o:.2f} (Frantz 1.06); radar adds {o - a:+.2f} m RMSE to optical (Frantz 0.45 m)")

# ---------------- 2. within-city SVR (median tuned settings per feature set) ----------------
rows = []
for c in sorted(np.unique(city)):
    m = city == c
    yc, fc = y[m], fold[m]
    rr = dict(city=c, n=int(m.sum()))
    for name in ["radar only", "optical only"]:
        X = tr.loc[m, SETS[name]].to_numpy(np.float32)
        p = np.full(m.sum(), np.nan)
        for k in np.unique(fc):
            trn, val = fc != k, fc == k
            if trn.sum() < 50 or not val.any():
                continue
            p[val] = np.clip(svr_pipe(**best[name]).fit(X[trn], yc[trn]).predict(X[val]), 0, None)
        ok = ~np.isnan(p)
        rr[f"{name}, within city"] = macro_rmse(yc[ok], p[ok])
        rr[f"{name}, pooled model"] = macro_rmse(yc, oof[name][m])
    rr["radar/optical, within city"] = rr["radar only, within city"] / rr["optical only, within city"]
    rows.append(rr)
    print(f"  {c}: radar {rr['radar only, within city']:.2f} | optical {rr['optical only, within city']:.2f} m")
wc = pd.DataFrame(rows).set_index("city")
wc.round(3).to_csv(f"{OUT}/within_city_svr.csv")
print("\n2. Within-city SVR, macro RMSE (m) and radar/optical ratio (Frantz, Berlin: 1.06 on RMSE)")
print(wc.round(2).to_string())
print(f"\nSaved in {OUT}: pooled_cv_svr.csv, within_city_svr.csv, parts/")

## 5. Leave-one-city-out

In [ ]:
# =====================================================================
# LEAVE-ONE-CITY-OUT CELL (CMP263): how well does the final model do in a city it has never seen?
# Frantz et al. (2021) leave-one-site-out: RMSE 3.83 to 8.14 m, weighted RMSE 2.92 to 3.55 m.
# Self-contained notebook cell. Resumable: each city's predictions are saved as soon as it finishes.
#
# For each of the 8 cities: the final design (mean of HistGB and SVR, same pipelines as the tuning
# cell: median imputer, scaler, Frantz step-1 RF selection for SVR) is trained on the TRAINING rows of
# the other 7 cities only, with the tuned final hyper-parameters held FIXED (re-tuning inside each
# run would take hours; state this simplification). The held-out city is then scored on
#   (a) its own training rows (height-stratified, includes tall buildings): macro RMSE, RMSE, bias,
#       RMSE per class, Frantz frequency-weighted RMSE (weights = the city's natural test histogram);
#   (b) its 1,100 natural test points: RMSE, MAE, bias, macro RMSE (with class sizes).
# For comparison, the same city's results with the city IN training are read when available:
#   (a) from the 5-fold CV of the final design (tuning_8cities/oof_tuned_long.csv, "mean HistGB+SVR"),
#   (b) from the final model on the test set (final_model/test_metrics.csv).
#
# =====================================================================
import os, json, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

DATA = config.OUT
TABLE = str(config.TABLE)
FINAL = f"{DATA}/final_model"
CV_OOF = f"{DATA}/tuning_8cities/oof_tuned_long.csv"
OUT = f"{DATA}/loco"
SEED = 42
CLASS_EDGES = [2, 6, 15, 30]
CLASS_NAMES = ["<2 m", "2-6 m", "6-15 m", "15-30 m", ">30 m"]
FINE_EDGES = [2, 4, 6, 9, 12, 15, 20, 30, 50]
os.makedirs(f"{OUT}/parts", exist_ok=True)


class RFThreshold(BaseEstimator, TransformerMixin):
    """Frantz et al. (2021) supplement A2, step 1 (same code as the tuning cell)."""
    def __init__(self, frac=0.10, n_trees=500, mtry=0.01, thr=1e-3, min_keep=20, min_rows=500, seed=0, n_jobs=1):
        self.frac, self.n_trees, self.mtry, self.thr = frac, n_trees, mtry, thr
        self.min_keep, self.min_rows, self.seed, self.n_jobs = min_keep, min_rows, seed, n_jobs

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        idx = rng.choice(len(y), size=min(len(y), max(int(self.frac * len(y)), self.min_rows)), replace=False)
        rf = RandomForestRegressor(n_estimators=self.n_trees, max_features=self.mtry, n_jobs=self.n_jobs,
                                   random_state=self.seed).fit(X[idx], y[idx])
        keep = np.where(rf.feature_importances_ > self.thr)[0]
        if len(keep) < self.min_keep:
            keep = np.argsort(rf.feature_importances_)[::-1][:self.min_keep]
        self.keep_ = np.sort(keep)
        return self

    def transform(self, X):
        return X[:, self.keep_]


def macro_rmse(y, p):
    c = np.digitize(y, CLASS_EDGES)
    return float(np.mean([np.sqrt(np.mean((p[c == k] - y[c == k]) ** 2)) for k in np.unique(c)]))


def weighted_rmse(y, p, frac):
    fine = np.digitize(y, FINE_EDGES)
    mse = [np.mean((p[fine == b] - y[fine == b]) ** 2) for b in range(len(FINE_EDGES) + 1) if (fine == b).any() and frac[b] > 0]
    w = [frac[b] for b in range(len(FINE_EDGES) + 1) if (fine == b).any() and frac[b] > 0]
    return float(np.sqrt(np.average(mse, weights=w))) if mse else np.nan


def scores(y, p, frac=None):
    e = p - y
    c = np.digitize(y, CLASS_EDGES)
    r = dict(n=len(y), RMSE_macro=macro_rmse(y, p), RMSE=float(np.sqrt(np.mean(e ** 2))), MAE=float(np.mean(np.abs(e))),
             bias=float(e.mean()))
    if frac is not None:
        r["RMSE_weighted"] = weighted_rmse(y, p, frac)
    for k, nm in enumerate(CLASS_NAMES):
        r[f"n {nm}"] = int((c == k).sum())
        r[f"RMSE {nm}"] = float(np.sqrt(np.mean(e[c == k] ** 2))) if (c == k).any() else np.nan
    return r


# ---------------- data and fixed final settings ----------------
META_COLS = {"sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split",
             "system:index", ".geo", "n_parts_50m", "slice_lo"}
df = config.read_table()
feats = [c for c in df.columns if c not in META_COLS]
df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
tr = df[df.split == "train"].dropna(subset=["height_m"]).reset_index(drop=True)
te = df[df.split == "test"].dropna(subset=["height_m"]).reset_index(drop=True)
del df
hp_h = {k.replace("model__", ""): v for k, v in json.load(open(f"{FINAL}/HistGB_params.json"))["best_params"].items()}
hp_s = {k.replace("model__regressor__", ""): v for k, v in json.load(open(f"{FINAL}/SVR_params.json"))["best_params"].items()}
print(f"HistGB {hp_h}\nSVR {hp_s}")


def pipelines():
    pre = [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
    h = Pipeline(pre + [("model", HistGradientBoostingRegressor(early_stopping=False, random_state=SEED, **hp_h))])
    s = Pipeline(pre + [("sel", RFThreshold(n_trees=500, seed=SEED, n_jobs=-1)),
                        ("model", TransformedTargetRegressor(regressor=SVR(kernel="rbf", **hp_s),
                                                            transformer=StandardScaler()))])
    return h, s


cities = sorted(tr.city.unique())
for c in cities:
    part = f"{OUT}/parts/{c}.csv"
    if os.path.exists(part):
        print(f"{c}: already done"); continue
    t0 = time.time()
    m_tr = tr.city != c
    X, y = tr.loc[m_tr, feats].to_numpy(np.float32), tr.loc[m_tr, "height_m"].to_numpy(float)
    h, s = pipelines()
    h.fit(X, y); s.fit(X, y)
    rows = []
    for setname, d in [("train rows (stratified)", tr[tr.city == c]), ("natural test", te[te.city == c])]:
        Xd = d[feats].to_numpy(np.float32)
        ph, ps = np.clip(h.predict(Xd), 0, None), np.clip(s.predict(Xd), 0, None)
        rows.append(pd.DataFrame({"sample_id": d.sample_id, "city": c, "set": setname, "y": d.height_m,
                                  "HistGB": ph, "SVR": ps, "mean": (ph + ps) / 2}))
    pd.concat(rows).to_csv(part, index=False)
    print(f"{c}: held out, trained on {m_tr.sum():,} rows of the other cities | "
          f"{len(s.named_steps['sel'].keep_)} SVR features | {(time.time() - t0) / 60:.1f} min")

# ---------------- metrics ----------------
P = pd.concat([pd.read_csv(f"{OUT}/parts/{c}.csv") for c in cities], ignore_index=True)
frac = {c: np.bincount(np.digitize(te.loc[te.city == c, "height_m"], FINE_EDGES), minlength=len(FINE_EDGES) + 1)
        / max((te.city == c).sum(), 1) for c in cities}
cv = pd.read_csv(CV_OOF) if os.path.exists(CV_OOF) else None
if cv is not None:
    cv = cv[cv.model == "mean HistGB+SVR"]
tm = pd.read_csv(f"{FINAL}/test_metrics.csv") if os.path.exists(f"{FINAL}/test_metrics.csv") else None

rows = []
for c in cities:
    a = P[(P.city == c) & (P.set == "train rows (stratified)")]
    b = P[(P.city == c) & (P.set == "natural test")]
    r = dict(city=c)
    for k, v in scores(a.y.to_numpy(), a["mean"].to_numpy(), frac[c]).items():
        r[f"LOCO train-rows {k}"] = v
    for k, v in scores(b.y.to_numpy(), b["mean"].to_numpy()).items():
        r[f"LOCO test {k}"] = v
    if cv is not None:
        g = cv[cv.city == c]
        r["5-fold train-rows RMSE_macro"] = macro_rmse(g.y.to_numpy(), g.pred.to_numpy())
        r["5-fold train-rows RMSE_weighted"] = weighted_rmse(g.y.to_numpy(), g.pred.to_numpy(), frac[c])
    if tm is not None:
        t = tm[(tm.model == "mean HistGB+SVR (final)") & (tm.subset == c)]
        if len(t):
            r["city-in test RMSE"], r["city-in test bias"] = float(t.RMSE.iloc[0]), float(t.bias.iloc[0])
    rows.append(r)
res = pd.DataFrame(rows).set_index("city")
res.round(3).to_csv(f"{OUT}/loco_metrics.csv")

show = [c for c in ["5-fold train-rows RMSE_macro", "LOCO train-rows RMSE_macro", "5-fold train-rows RMSE_weighted",
                    "LOCO train-rows RMSE_weighted", "LOCO train-rows bias", "LOCO train-rows RMSE >30 m",
                    "city-in test RMSE", "LOCO test RMSE", "city-in test bias", "LOCO test bias"] if c in res]
print("\nLeave-one-city-out vs city in training (mean HistGB+SVR, metres).")
print("train-rows = the held-out city's stratified training rows; test = its 1,100 natural test points.")
print(res[show].round(2).to_string())
a = P[P.set == "train rows (stratified)"]
print(f"\nAll held-out training rows pooled: macro RMSE {macro_rmse(a.y.to_numpy(), a['mean'].to_numpy()):.2f} m, "
      f"RMSE {np.sqrt(np.mean((a['mean'] - a.y) ** 2)):.2f} m, weighted RMSE (mean over cities) "
      f"{res['LOCO train-rows RMSE_weighted'].mean():.2f} m")
b = P[P.set == "natural test"]
print(f"All natural test points: RMSE {np.sqrt(np.mean((b['mean'] - b.y) ** 2)):.2f} m, bias {(b['mean'] - b.y).mean():+.2f} m "
      f"(Frantz leave-one-site-out: RMSE 3.83 to 8.14 m, weighted 2.92 to 3.55 m)")

# ---------------- figure ----------------
if "5-fold train-rows RMSE_macro" in res:
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
    xx = np.arange(len(res))
    for ax, (a_, b_, lab) in zip(axs, [("5-fold train-rows RMSE_macro", "LOCO train-rows RMSE_macro", "macro RMSE, stratified rows (m)"),
                                       ("city-in test RMSE", "LOCO test RMSE", "RMSE, natural test points (m)")]):
        if a_ not in res or b_ not in res:
            ax.axis("off"); continue
        ax.bar(xx - 0.2, res[a_], 0.4, color="#bdbdbd", label="city in training")
        ax.bar(xx + 0.2, res[b_], 0.4, color="#2a6fdb", label="city left out")
        ax.set_xticks(xx); ax.set_xticklabels([c.replace("_", " ") for c in res.index], rotation=20, fontsize=8)
        ax.set_ylabel(lab); ax.legend(frameon=False, fontsize=8)
        for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
    fig.suptitle("Cost of predicting an unseen city (mean of HistGB and SVR)", fontsize=10)
    fig.tight_layout(); fig.savefig(f"{OUT}/loco.png", dpi=200); plt.show()
print(f"\nSaved in {OUT}: parts/<city>.csv, loco_metrics.csv, loco.png")